**Test functions in the convnet training flow**
- Folder name can be set according to your need.
- For the small model, your output file /result/F32B3R1_zebra_test.png should be similar to /reference/F32B3R1_zebra_test_golden.png **(PSNR>40dB)** to get a full score.
- For the large model, your output file /result/F64B8R2_zebra_test.png should be similar to /reference/F64B8R2_zebra_test_golden.png **(PSNR>40dB)** to get a full score.

In [ ]:
from pathlib import Path
import os


def find_hw03_root():
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / 'optimization-based').exists() and (path / 'convnet-based').exists():
            return Path(os.path.relpath(path, cwd))

        hw03 = path / 'hw03'
        if (hw03 / 'optimization-based').exists() and (hw03 / 'convnet-based').exists():
            return Path(os.path.relpath(hw03, cwd))

    raise FileNotFoundError('Run this notebook from the repository root, hw03, or an hw03 subfolder.')


FOLDER_NAME = str(find_hw03_root())


Do not modify the remaining code.

In [ ]:
%cd '{FOLDER_NAME}/convnet-based'
%run 'model.ipynb'

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.autograd import Variable
import imageio.v2 as imageio
import numpy as np
from types import SimpleNamespace
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric

**Testing on the small model**
- Make sure that the PSNR with respect to TAs reference golden image is higher than **40 dB**.

In [ ]:
# ===== Testing settings: small model =====#
args = SimpleNamespace(
    model_path = "model_trained/net_F32B3R1_epoch_15.pth",
    input_image_path = "image_test/LR_zebra_test.png",
    output_image_path = "result/F32B3R1_zebra_test.png",
    reference_image_path = "reference/F32B3R1_zebra_test_golden.png",
    HR_image_path = "reference/HR_zebra_test.png",
    cuda=torch.cuda.is_available()
)

print(args)

if args.cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

In [ ]:
#===== load ZebraSRNet model =====#
print('===> Loading model')
net = torch.load(args.model_path, map_location=torch.device('cpu'), weights_only=False)
if args.cuda:
    net = net.cuda()

#===== Load input image =====#
imgIn = imageio.imread(args.input_image_path)/255.0
imgIn = imgIn.transpose((2,0,1)).astype(float)
imgIn = imgIn.reshape(1, *imgIn.shape)
imgIn = torch.Tensor(imgIn)

#===== Test procedures =====#
varIn = Variable(imgIn)
if args.cuda:
    varIn = varIn.cuda()

prediction = net(varIn)
prediction = prediction.data.cpu().numpy().squeeze().transpose((1,2,0))
img_out = np.round(255*np.clip(prediction, 0.0, 1.0)).astype('uint8')
imageio.imwrite(args.output_image_path, img_out)

def check_ans(img_urs_path, img_ref_path):
    img_urs = imageio.imread(img_urs_path)
    img_ref = imageio.imread(img_ref_path)
    psnr = psnr_metric(img_ref, img_urs)
    if np.isinf(psnr):
        print("===> PSNR: inf dB (images are identical)")
    else:
        print("===> PSNR: {:.4f} dB".format(psnr))
    ssim = ssim_metric(img_ref, img_urs, channel_axis=-1, data_range=255)
    print('===> SSIM: {:.4f}'.format(ssim))

# check answer
print('===> Test for small model: F32B3R1')
print('===== compare with original HR image ===== ')
check_ans(args.output_image_path, args.HR_image_path)

img_ref_path =  f'{FOLDER_NAME}/optimization-based/reference/zebra_test_single_golden.png'
print('\n===== compare with TAs reference answer ===== ')
check_ans(args.output_image_path, args.reference_image_path)

**Testing on the large model**
- Make sure that the PSNR with respect to TAs reference golden image is higher than **40 dB**.

In [ ]:
# ===== Testing settings: large model =====#
args = SimpleNamespace(
    model_path = "model_trained/net_F64B8R2_epoch_120.pth",
    input_image_path = "image_test/LR_zebra_test.png",
    output_image_path = "result/F64B8R2_zebra_test.png",
    reference_image_path = "reference/F64B8R2_zebra_test_golden.png",
    HR_image_path = "reference/HR_zebra_test.png",
    cuda=torch.cuda.is_available()
)

print(args)

if args.cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

In [ ]:
#===== load ZebraSRNet model =====#
print('===> Loading model')
net = torch.load(args.model_path, map_location=torch.device('cpu'), weights_only=False)
if args.cuda:
    net = net.cuda()

#===== Load input image =====#
imgIn = imageio.imread(args.input_image_path)/255.0
imgIn = imgIn.transpose((2,0,1)).astype(float)
imgIn = imgIn.reshape(1, *imgIn.shape)
imgIn = torch.Tensor(imgIn)

#===== Test procedures =====#
varIn = Variable(imgIn)
if args.cuda:
    varIn = varIn.cuda()

prediction = net(varIn)
prediction = prediction.data.cpu().numpy().squeeze().transpose((1,2,0))
img_out = np.round(255*np.clip(prediction, 0.0, 1.0)).astype('uint8')
imageio.imwrite(args.output_image_path, img_out)

def check_ans(img_urs_path, img_ref_path):
    img_urs = imageio.imread(img_urs_path)
    img_ref = imageio.imread(img_ref_path)
    psnr = psnr_metric(img_ref, img_urs)
    if np.isinf(psnr):
        print("===> PSNR: inf dB (images are identical)")
    else:
        print("===> PSNR: {:.4f} dB".format(psnr))
    ssim = ssim_metric(img_ref, img_urs, channel_axis=-1, data_range=255)
    print('===> SSIM: {:.4f}'.format(ssim))

# check answer
print('===> Test for large model: F64B8R2')
print('===== compare with original HR image ===== ')
check_ans(args.output_image_path, args.HR_image_path)

img_ref_path =  f'{FOLDER_NAME}/optimization-based/reference/zebra_test_single_golden.png'
print('\n===== compare with TAs reference answer ===== ')
check_ans(args.output_image_path, args.reference_image_path)


**Testing on the large model using my own dataset**

In [ ]:
# ===== Testing settings: large model =====#
args = SimpleNamespace(
    model_path = "model_trained/own_net_F64B8R2_epoch_120.pth",
    input_image_path = "own_image/LR_test.png",
    output_image_path = "own_image/F64B8R2_own_test.png",
    HR_image_path = "own_image/HR_test.png",
    cuda=torch.cuda.is_available()
)

print(args)

if args.cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

In [ ]:
#===== load ZebraSRNet model =====#
print('===> Loading model')
net = torch.load(args.model_path, map_location=torch.device('cpu'), weights_only=False)
if args.cuda:
    net = net.cuda()

#===== Load input image =====#
imgIn = imageio.imread(args.input_image_path)/255.0
imgIn = imgIn.transpose((2,0,1)).astype(float)
imgIn = imgIn.reshape(1, *imgIn.shape)
imgIn = torch.Tensor(imgIn)

#===== Test procedures =====#
varIn = Variable(imgIn)
if args.cuda:
    varIn = varIn.cuda()

prediction = net(varIn)
prediction = prediction.data.cpu().numpy().squeeze().transpose((1,2,0))
img_out = np.round(255*np.clip(prediction, 0.0, 1.0)).astype('uint8')
imageio.imwrite(args.output_image_path, img_out)

def check_ans(img_urs_path, img_ref_path):
    img_urs = imageio.imread(img_urs_path)
    img_ref = imageio.imread(img_ref_path)
    psnr = psnr_metric(img_ref, img_urs)
    if np.isinf(psnr):
        print("===> PSNR: inf dB (images are identical)")
    else:
        print("===> PSNR: {:.4f} dB".format(psnr))
    ssim = ssim_metric(img_ref, img_urs, channel_axis=-1, data_range=255)
    print('===> SSIM: {:.4f}'.format(ssim))

# check answer
print('===> Test for large model: F64B8R2')
print('===== compare with original HR image ===== ')
check_ans(args.output_image_path, args.HR_image_path)
